## Gold Layer - Business Analytics
#### Architecture
- Bronze Layer: Raw data ingestion from CSV files
- Silver Layer: Cleaned and validated data (5 tables)
- Gold Layer: Business-ready aggregated analytics (4 tables)

In [0]:
     
%sql show tables in olist_silver

database,tableName,isTemporary
olist_silver,customers,false
olist_silver,order_items,false
olist_silver,orders,false
olist_silver,payments,false
olist_silver,products,false


In [0]:
%sql CREATE DATABASE IF NOT EXISTS olist_gold


In [0]:
%sql USE olist_gold

In [0]:
%sql
-- Customer Metrics: Aggregates customer purchase behavior, lifetime value, and segments customers by engagement level

CREATE OR REPLACE TABLE olist_gold.customer_metrics
USING DELTA
AS
SELECT
    c.customer_id,
    c.customer_state,
    c.customer_city,
    COUNT(DISTINCT o.order_id) AS total_orders,
    SUM(p.payment_value) AS total_revenue,
    ROUND(AVG(p.payment_value), 2) AS avg_order_value,
    MIN(o.order_purchase_ts) AS first_order_date,
    MAX(o.order_purchase_ts) AS last_order_date,
    DATEDIFF(MAX(o.order_purchase_ts), MIN(o.order_purchase_ts)) AS customer_lifetime_days,
    CASE 
        WHEN COUNT(DISTINCT o.order_id) >= 5 THEN 'Loyal'
        WHEN COUNT(DISTINCT o.order_id) >= 3 THEN 'Regular'
        WHEN COUNT(DISTINCT o.order_id) >= 2 THEN 'Repeat'
        ELSE 'One-time'
    END AS customer_segment
FROM olist_silver.customers c
LEFT JOIN olist_silver.orders o ON c.customer_id = o.customer_id
LEFT JOIN olist_silver.payments p ON o.order_id = p.order_id
GROUP BY c.customer_id, c.customer_state, c.customer_city;

num_affected_rows,num_inserted_rows


In [0]:
%sql
-- Product Performance: Analyzes product and category sales performance with revenue rankings for identifying top performers 
CREATE OR REPLACE TABLE olist_gold.product_performance
USING DELTA
AS
WITH product_stats AS (
    SELECT
        pr.product_id,
        pr.product_category_name,
        COUNT(DISTINCT oi.order_id) AS total_orders,
        COUNT(oi.order_item_id) AS total_units_sold,
        SUM(oi.price) AS total_revenue,
        ROUND(AVG(oi.price), 2) AS avg_price,
        SUM(oi.freight_value) AS total_freight
    FROM olist_silver.products pr
    LEFT JOIN olist_silver.order_items oi ON pr.product_id = oi.product_id
    GROUP BY pr.product_id, pr.product_category_name
)
SELECT 
    *,
    ROW_NUMBER() OVER (ORDER BY total_revenue DESC) AS revenue_rank,
    ROW_NUMBER() OVER (PARTITION BY product_category_name ORDER BY total_revenue DESC) AS category_rank
FROM product_stats;

num_affected_rows,num_inserted_rows


In [0]:
%sql select * from olist_silver.orders limit 2

order_id,customer_id,order_status,order_purchase_ts,order_deliverd_ts
ccbabeb0b02433bd0fcbac46e70339f2,c77ee2d8ba1614a4d489a44166894938,delivered,2018-02-19T20:31:09.000Z,2018-03-09T22:22:25.000Z
c6bf92017bd40729c135b58b643f64c2,3d3c463710ea6e8dd9a63c1110eeb06b,delivered,2018-08-08T01:15:06.000Z,2018-08-15T00:18:43.000Z


In [0]:
%sql select count(*) from olist_silver.orders where order_status = 'delivered'

count(*)
96478


In [0]:
%sql
-- Orders Fact: Denormalized fact table combining all order dimensions and metrics for efficient analytical queries
CREATE OR REPLACE TABLE olist_gold.orders_fact
USING DELTA
AS
SELECT
    -- Order dimensions
    o.order_id,
    o.order_status,
    o.order_purchase_ts,
    o.order_deliverd_ts,
    DATE(o.order_purchase_ts) AS order_date,
    YEAR(o.order_purchase_ts) AS order_year,
    MONTH(o.order_purchase_ts) AS order_month,
    
    -- Customer dimensions
    c.customer_id,
    c.customer_city,
    c.customer_state,
    
    -- Product dimensions
    pr.product_id,
    pr.product_category_name,
    
    -- Order item metrics
    oi.order_item_id,
    oi.price AS item_price,
    oi.freight_value,
    
    -- Payment metrics
    p.payment_type,
    p.payment_value,
    
    -- Calculated metrics
    DATEDIFF(o.order_deliverd_ts, o.order_purchase_ts) AS delivery_days,
    oi.price + oi.freight_value AS total_item_cost
    
FROM olist_silver.orders o
INNER JOIN olist_silver.order_items oi ON o.order_id = oi.order_id
LEFT JOIN olist_silver.customers c ON o.customer_id = c.customer_id
LEFT JOIN olist_silver.products pr ON oi.product_id = pr.product_id
LEFT JOIN olist_silver.payments p ON o.order_id = p.order_id
WHERE o.order_status = 'delivered';

num_affected_rows,num_inserted_rows


In [0]:
%sql
-- Monthly Sales Trends: Tracks monthly revenue trends with month-over-month growth analysis for business performance monitoring
CREATE OR REPLACE TABLE olist_gold.monthly_sales_trends
USING DELTA
AS
SELECT
    DATE_TRUNC('MONTH', o.order_purchase_ts) AS month,
    COUNT(DISTINCT o.order_id) AS total_orders,
    COUNT(DISTINCT o.customer_id) AS unique_customers,
    SUM(p.payment_value) AS total_revenue,
    ROUND(AVG(p.payment_value), 2) AS avg_order_value,
    
    -- Growth metrics
    LAG(SUM(p.payment_value)) OVER (ORDER BY DATE_TRUNC('MONTH', o.order_purchase_ts)) AS prev_month_revenue,
    ROUND(
        (SUM(p.payment_value) - LAG(SUM(p.payment_value)) OVER (ORDER BY DATE_TRUNC('MONTH', o.order_purchase_ts))) 
        / LAG(SUM(p.payment_value)) OVER (ORDER BY DATE_TRUNC('MONTH', o.order_purchase_ts)) * 100, 
        2
    ) AS revenue_growth_pct
    
FROM olist_silver.orders o
LEFT JOIN olist_silver.payments p ON o.order_id = p.order_id
GROUP BY DATE_TRUNC('MONTH', o.order_purchase_ts)
ORDER BY month;

num_affected_rows,num_inserted_rows


In [0]:
%sql
-- Check table counts
SELECT 'customer_metrics' as table_name, COUNT(*) as row_count 
FROM olist_gold.customer_metrics
UNION ALL
SELECT 'product_performance', COUNT(*) FROM olist_gold.product_performance
UNION ALL
SELECT 'orders_fact', COUNT(*) FROM olist_gold.orders_fact
UNION ALL
SELECT 'monthly_sales_trends', COUNT(*) FROM olist_gold.monthly_sales_trends;

table_name,row_count
customer_metrics,99441
product_performance,32951
orders_fact,104680
monthly_sales_trends,25


In [0]:
%sql
-- Optimize tables with Z-ordering for faster queries
OPTIMIZE olist_gold.customer_metrics ZORDER BY (customer_id);
OPTIMIZE olist_gold.product_performance ZORDER BY (product_category_name);
OPTIMIZE olist_gold.orders_fact ZORDER BY (order_date, customer_id);
OPTIMIZE olist_gold.monthly_sales_trends ZORDER BY (month);

path,metrics
,"List(0, 0, List(null, null, 0.0, 0, 0), List(null, null, 0.0, 0, 0), 0, List(minCubeSize(107374182400), List(0, 0), List(1, 3079), 0, List(0, 0), 0, null), null, 0, 0, 1, 1, false, 0, 0, 1769661310111, 1769661310516, 8, 0, null, List(0, 0), null, 7, 7, 0, 0, null)"


In [0]:
%sql
-- Add table constraints to ensure data quality
ALTER TABLE olist_gold.customer_metrics 
ADD CONSTRAINT positive_orders CHECK (total_orders > 0);



In [0]:
%sql
ALTER TABLE olist_gold.orders_fact 
ADD CONSTRAINT valid_payment CHECK (payment_value >= 0);

In [0]:
%sql
-- Check table properties
DESCRIBE DETAIL olist_gold.customer_metrics;


format,id,name,description,location,createdAt,lastModified,partitionColumns,clusteringColumns,numFiles,sizeInBytes,properties,minReaderVersion,minWriterVersion,tableFeatures,statistics,clusterByAuto
delta,ff9dda75-4fa2-4ab8-94e3-b3b22b96cb36,workspace.olist_gold.customer_metrics,null,,2026-01-29T03:48:04.495Z,2026-01-29T03:48:07.000Z,List(),List(),1,3928204,"Map(delta.parquet.compression.codec -> zstd, delta.enableDeletionVectors -> true)",3,7,"List(appendOnly, deletionVectors, invariants)","Map(numRowsDeletedByDeletionVectors -> 0, numDeletionVectors -> 0)",false


In [0]:
%sql DESCRIBE HISTORY olist_gold.customer_metrics;

version,timestamp,userId,userName,operation,operationParameters,job,notebook,clusterId,readVersion,isolationLevel,isBlindAppend,operationMetrics,userMetadata,engineInfo
2,2026-01-29T03:48:07.000Z,72353017163422,guna.thangavel36@gmail.com,CREATE OR REPLACE TABLE AS SELECT,"Map(partitionBy -> [], clusterBy -> [], description -> null, isManaged -> true, properties -> {""delta.parquet.compression.codec"":""zstd"",""delta.enableDeletionVectors"":""true""}, statsOnLoad -> true)",null,List(2841511027769394),0129-031149-gs9bh3xq-v2n,1,WriteSerializable,false,"Map(numFiles -> 1, numRemovedFiles -> 1, numRemovedBytes -> 3721997, numDeletionVectorsRemoved -> 0, numOutputRows -> 99441, numOutputBytes -> 3928204)",null,Databricks-Runtime/17.3.x-aarch64-photon-scala2.13
1,2026-01-29T03:19:03.000Z,72353017163422,guna.thangavel36@gmail.com,CREATE OR REPLACE TABLE AS SELECT,"Map(partitionBy -> [], clusterBy -> [], description -> null, isManaged -> true, properties -> {""delta.parquet.compression.codec"":""zstd"",""delta.enableDeletionVectors"":""true""}, statsOnLoad -> true)",null,List(2841511027769394),0129-031149-gs9bh3xq-v2n,0,WriteSerializable,false,"Map(numFiles -> 1, numRemovedFiles -> 1, numRemovedBytes -> 3721997, numDeletionVectorsRemoved -> 0, numOutputRows -> 99441, numOutputBytes -> 3721997)",null,Databricks-Runtime/17.3.x-aarch64-photon-scala2.13
0,2026-01-29T03:15:17.000Z,72353017163422,guna.thangavel36@gmail.com,CREATE OR REPLACE TABLE AS SELECT,"Map(partitionBy -> [], clusterBy -> [], description -> null, isManaged -> true, properties -> {""delta.parquet.compression.codec"":""zstd"",""delta.enableDeletionVectors"":""true""}, statsOnLoad -> true)",null,List(2841511027769394),0129-031149-gs9bh3xq-v2n,null,WriteSerializable,false,"Map(numFiles -> 1, numRemovedFiles -> 0, numRemovedBytes -> 0, numDeletionVectorsRemoved -> 0, numOutputRows -> 99441, numOutputBytes -> 3721997)",null,Databricks-Runtime/17.3.x-aarch64-photon-scala2.13


In [0]:
%sql
-- Clean up old file versions (keeps last 7 days)
-- Only run if you want to save storage space
VACUUM olist_gold.customer_metrics RETAIN 168 HOURS;
VACUUM olist_gold.orders_fact RETAIN 168 HOURS;

path
""


In [0]:
%sql
-- Create business views for common analytics queries
-- Top customers view (Loyal and Regular customers only)
CREATE OR REPLACE VIEW olist_gold.vw_top_customers AS
SELECT * FROM olist_gold.customer_metrics
WHERE customer_segment IN ('Loyal', 'Regular');

-- Top selling products view(Top 50 by revenue)
CREATE OR REPLACE VIEW olist_gold.vw_top_products AS
SELECT * FROM olist_gold.product_performance
WHERE revenue_rank <= 50;

-- Revenue by State
CREATE OR REPLACE VIEW olist_gold.vw_revenue_by_state AS
SELECT
  customer_state,
  SUM(total_revenue) AS total_revenue,
  COUNT(*) AS customer_count
FROM olist_gold.customer_metrics
GROUP BY customer_state;

--Monthly Sales Trend
CREATE OR REPLACE VIEW olist_gold.vw_monthly_sales_trend AS
SELECT
  month,
  total_orders,
  total_revenue,
  avg_order_value
FROM olist_gold.monthly_sales_trends;

--Payment Type Distribution
CREATE OR REPLACE VIEW olist_gold.vw_payment_distribution AS
SELECT
  payment_type,
  COUNT(DISTINCT order_id) AS total_orders,
  SUM(payment_value) AS total_revenue
FROM olist_gold.orders_fact
GROUP BY payment_type;

--Delivery Performance Buckets
CREATE OR REPLACE VIEW olist_gold.vw_delivery_performance AS
SELECT
  CASE
    WHEN delivery_days <= 7 THEN 'Fast'
    WHEN delivery_days <= 14 THEN 'Normal'
    WHEN delivery_days <= 21 THEN 'Slow'
    ELSE 'Very Slow'
  END AS delivery_bucket,
  COUNT(*) AS order_count,
  AVG(delivery_days) AS avg_delivery_days
FROM olist_gold.orders_fact
GROUP BY
  CASE
    WHEN delivery_days <= 7 THEN 'Fast'
    WHEN delivery_days <= 14 THEN 'Normal'
    WHEN delivery_days <= 21 THEN 'Slow'
    ELSE 'Very Slow'
  END;

-- KPI view
CREATE OR REPLACE VIEW olist_gold.vw_kpi_summary AS
SELECT
  ROUND(SUM(payment_value), 2)        AS total_revenue,
  COUNT(DISTINCT order_id)            AS total_orders,
  ROUND(
    SUM(payment_value) / COUNT(DISTINCT order_id), 2
  )                                   AS avg_order_value
FROM olist_gold.orders_fact;

#### Gold Layer Tables
1. customer_metrics: Customer behavior and segmentation
2. product_performance: Product sales rankings
3. orders_fact: Denormalized order analytics
4. monthly_sales_trends: Time-series revenue analysis

#### Key Features Implemented
- Delta Lake for ACID transactions
- Table optimization with Z-ordering
- Data quality constraints
- Business intelligence views